In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!rm -rf /content/project_dataset
!cp -r "/content/drive/MyDrive/project_dataset" /content/

In [3]:
import os

test_path = "/content/project_dataset/test"

print("Before Fix:", os.listdir(test_path))

# Remove hidden/system folders
for folder in os.listdir(test_path):
    if folder.startswith('.'):
        os.system(f'rm -rf "{test_path}/{folder}"')

print("After Fix:", os.listdir(test_path))

Before Fix: ['Surti', 'Other', 'Khillari', 'Gir', 'Deoni', 'Nagpuri', 'Murrah', 'Jersey_Crossbred', 'Pandharpuri', 'Human']
After Fix: ['Surti', 'Other', 'Khillari', 'Gir', 'Deoni', 'Nagpuri', 'Murrah', 'Jersey_Crossbred', 'Pandharpuri', 'Human']


In [5]:
print("Train:", len(os.listdir("/content/project_dataset/train")))
print("Val:", len(os.listdir("/content/project_dataset/Validation")))
print("Test:", len(os.listdir("/content/project_dataset/test")))

Train: 10
Val: 10
Test: 10


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.3,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7,1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [7]:
IMG_SIZE = (160,160)
BATCH_SIZE = 32

train_data = train_datagen.flow_from_directory(
    "/content/project_dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    "/content/project_dataset/Validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    "/content/project_dataset/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_mode='categorical'
)

Found 2420 images belonging to 10 classes.
Found 396 images belonging to 10 classes.
Found 396 images belonging to 10 classes.


In [8]:
print(train_data.num_classes)
print(val_data.num_classes)
print(test_data.num_classes)

print(train_data.class_indices)
print(test_data.class_indices)

10
10
10
{'Deoni': 0, 'Gir': 1, 'Human': 2, 'Jersey_Crossbred': 3, 'Khillari': 4, 'Murrah': 5, 'Nagpuri': 6, 'Other': 7, 'Pandharpuri': 8, 'Surti': 9}
{'Deoni': 0, 'Gir': 1, 'Human': 2, 'Jersey_Crossbred': 3, 'Khillari': 4, 'Murrah': 5, 'Nagpuri': 6, 'Other': 7, 'Pandharpuri': 8, 'Surti': 9}


In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_data.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))

In [10]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early = EarlyStopping(patience=3, restore_best_weights=True)
reduce = ReduceLROnPlateau(patience=2, factor=0.3)

In [13]:
history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=12,
    class_weight=class_weights,
    callbacks=[early, reduce]
)

Epoch 1/12
14/76 ━━━━━━━━━━━━━━━━━━━━ 36s 589ms/step - accuracy: 0.1678 - loss: 3.2194

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


76/76 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.4161 - loss: 2.0097 - val_accuracy: 0.6540 - val_loss: 1.0126 - learning_rate: 0.0010
Epoch 2/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 48s 637ms/step - accuracy: 0.5843 - loss: 1.2425 - val_accuracy: 0.6894 - val_loss: 0.8353 - learning_rate: 0.0010
Epoch 3/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 50s 664ms/step - accuracy: 0.6326 - loss: 1.0444 - val_accuracy: 0.7500 - val_loss: 0.7050 - learning_rate: 0.0010
Epoch 4/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 47s 624ms/step - accuracy: 0.6814 - loss: 0.9062 - val_accuracy: 0.7702 - val_loss: 0.6386 - learning_rate: 0.0010
Epoch 5/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 48s 635ms/step - accuracy: 0.7029 - loss: 0.8146 - val_accuracy: 0.7929 - val_loss: 0.5860 - learning_rate: 0.0010
Epoch 6/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 641ms/step - accuracy: 0.7087 - loss: 0.7826 - val_accuracy: 0.8030 - val_loss: 0.5741 - learning_rate: 0.0010
Epoch 7/12
76/76 ━━━━━━━━━━━━━━━━━━━━ 48s 630ms/step - accuracy: 0.7467 - loss: 0.6887 - val_ac

In [14]:
for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early, reduce]
)

Epoch 1/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 94s 930ms/step - accuracy: 0.6318 - loss: 1.0216 - val_accuracy: 0.8359 - val_loss: 0.4715 - learning_rate: 1.0000e-05
Epoch 2/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 49s 645ms/step - accuracy: 0.6789 - loss: 0.8460 - val_accuracy: 0.8258 - val_loss: 0.4751 - learning_rate: 1.0000e-05
Epoch 3/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 50s 658ms/step - accuracy: 0.6963 - loss: 0.8060 - val_accuracy: 0.8308 - val_loss: 0.4679 - learning_rate: 3.0000e-06
Epoch 4/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 51s 671ms/step - accuracy: 0.7087 - loss: 0.7507 - val_accuracy: 0.8232 - val_loss: 0.4606 - learning_rate: 3.0000e-06
Epoch 5/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 52s 682ms/step - accuracy: 0.7178 - loss: 0.7493 - val_accuracy: 0.8333 - val_loss: 0.4536 - learning_rate: 3.0000e-06
Epoch 6/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 50s 656ms/step - accuracy: 0.7186 - loss: 0.7330 - val_accuracy: 0.8258 - val_loss: 0.4506 - learning_rate: 3.0000e-06
Epoch 7/10
76/76 ━━━━━━━━━━━━━━━━━━━━ 50s 655ms/step - acc

In [15]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 290ms/step - accuracy: 0.8409 - loss: 0.4403
Test Accuracy: 0.8409090638160706


In [16]:
model.save("/content/drive/MyDrive/final_model.h5")